In [1]:
"""
# PhishGuard AI - Train and Compare Model

## Objective

This notebook trains and compares two machine learning classifiers:

1. Decision Tree Classifier
2. Random Forest Classifier

The models are evaluated using:

- Accuracy
- Precision
- Recall
- F1-score

The model with the highest phishing-class F1 score will be selected as the deployment model.

## Label Convention

- 0 = Legitimate
- 1 = Phishing
"""

'\n# PhishGuard AI - Train and Compare Model\n\n## Objective\n\nThis notebook trains and compares two machine learning classifiers:\n\n1. Decision Tree Classifier\n2. Random Forest Classifier\n\nThe models are evaluated using:\n\n- Accuracy\n- Precision\n- Recall\n- F1-score\n\nThe model with the highest phishing-class F1 score will be selected as the deployment model.\n\n## Label Convention\n\n- 0 = Legitimate\n- 1 = Phishing\n'

In [2]:

import json
import os
import string
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier



In [3]:

sys.path.insert(0, os.path.abspath("../backend"))
from app.features import (  # noqa: E402
    URL_ONLY_FEATURES,
    canonicalize_url,
    compute_features,
)


In [4]:
RANDOM_STATE = 42
CSV = "../PhiUSIIL_Phishing_URL_Dataset.csv"
OUT = "../outputs"
os.makedirs(OUT, exist_ok=True)

In [5]:
if not os.path.exists(CSV):
    raise SystemExit(f"Dataset not found: {CSV} (place PhiUSIIL CSV in repo root)")

df = pd.read_csv(CSV, encoding="utf-8").drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=["URL"]).reset_index(drop=True)
print(f"Loaded {len(df):,} unique rows from {CSV}")

# Targeted legitimate augmentation: PhiUSIIL's "legitimate" class is 100%
# bare homepage URLs (no path, no query string, no digits, always https --
# see manual_test_results.csv). Add real legitimate URLs that DO have a
# path / query string / plain http so the model sees that those are
# normal, not just a phishing tell.
AUG_CSV = "../legit_augmentation.csv"
if os.path.exists(AUG_CSV):
    aug = pd.read_csv(AUG_CSV, encoding="utf-8")
    before = len(df)
    df = df[["URL", "label"]].copy()
    df["_is_aug"] = 0
    aug = aug[["URL", "label"]].copy()
    aug["_is_aug"] = 1
    df = pd.concat([df, aug], ignore_index=True)
    df = df.drop_duplicates(subset=["URL"]).reset_index(drop=True)
    print(f"Augmented with {len(aug):,} curated legitimate URLs "
          f"({before:,} -> {len(df):,} total rows)")


Loaded 235,795 unique rows from ../PhiUSIIL_Phishing_URL_Dataset.csv
Augmented with 242 curated legitimate URLs (235,795 -> 235,612 total rows)


In [6]:
y = 1 - df["label"].astype(int)
print(f"Class balance after flip -> Legit(0): {int((y==0).sum()):,} | "
      f"Phish(1): {int((y==1).sum()):,}")

Class balance after flip -> Legit(0): 135,092 | Phish(1): 100,520


In [7]:
print("\nBuilding TLD lookup...")
tld_lookup = (
    df.assign(_legit=(y == 0).astype(int))
      .groupby(df["URL"].apply(
          lambda u: u.split("//")[-1].split("/")[0].split(":")[0].rsplit(".", 1)[-1].lower()
          if "." in u.split("//")[-1].split("/")[0] else ""
      ))["_legit"]
      .mean()
      .to_dict()
)
print(f"  {len(tld_lookup)} distinct TLDs")

# Character -> P(char | legit URL). Built from legitimate URLs only, with
# +1 smoothing so any character seen at inference has a non-zero prob.
#
# IMPORTANT: PhiUSIIL recorded 100% of legitimate URLs as "www.<domain>"
# (only ~41% of phishing URLs have www.), so counting raw URL characters
# inflates P('w') / P('.') and makes any bare-domain URL (chatgpt.com,
# github.com, ...) look less "legit-shaped" than it should. Canonicalize
# (strip www.) before counting so the table reflects real character
# distribution, not this labeling artifact -- same helper compute_features()
# uses at inference, so train/serve stay consistent.
print("\nBuilding character-frequency lookup from legit URLs...")
counter = {c: 1 for c in string.printable}
total = sum(counter.values())
legit_urls = df.loc[y == 0, "URL"].astype(str)
for u in legit_urls.sample(min(50_000, len(legit_urls)), random_state=RANDOM_STATE):
    for c in canonicalize_url(u).lower():
        counter[c] = counter.get(c, 0) + 1
        total += 1
char_lookup = {c: v / total for c, v in counter.items()}
print(f"  {len(char_lookup)} distinct characters, total mass = "
      f"{sum(char_lookup.values()):.4f}")

# Save the lookups so features.py can load them at inference
joblib.dump(tld_lookup, os.path.join(OUT, "tld_probability_lookup.joblib"))
joblib.dump(char_lookup, os.path.join(OUT, "char_probability_lookup.joblib"))


Building TLD lookup...


  710 distinct TLDs

Building character-frequency lookup from legit URLs...


  100 distinct characters, total mass = 1.0000


['../outputs\\char_probability_lookup.joblib']

In [8]:
print(f"\nExtracting features from {len(df):,} URLs using compute_features()...")
feats = [compute_features(u, tld_lookup, char_lookup) for u in df["URL"].astype(str)]
X = pd.DataFrame(feats)[URL_ONLY_FEATURES].astype(float)
print(f"  Feature matrix shape: {X.shape}")


Extracting features from 235,612 URLs using compute_features()...


  Feature matrix shape: (235612, 20)


In [9]:
AUG_WEIGHT = 25  # up-weight the curated legit augmentation rows
is_aug = df["_is_aug"].values if "_is_aug" in df.columns else np.zeros(len(df))

X_train, X_test, y_train, y_test, aug_train, aug_test = train_test_split(
    X, y, is_aug, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
sample_weight_train = np.where(aug_train == 1, AUG_WEIGHT, 1.0)
print("Train:", len(X_train), "| Test:", len(X_test),
      "| augmented rows in train:", int(aug_train.sum()), "(weight x" + str(AUG_WEIGHT) + ")")


def score(model, name):
    """Fit (up-weighting augmented rows), evaluate on held-out test set."""
    model.fit(X_train, y_train, sample_weight=sample_weight_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    m = {
        "name": name,
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, pos_label=1)),
        "recall": float(recall_score(y_test, pred, pos_label=1)),
        "f1_score": float(f1_score(y_test, pred, pos_label=1)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_features": len(URL_ONLY_FEATURES),
    }
    print(name)
    print(classification_report(
        y_test, pred, target_names=["Legitimate(0)", "Phishing(1)"], digits=4,
    ))
    return model, m



Train: 188489 | Test: 47123 | augmented rows in train: 188 (weight x25)


In [10]:
dt = DecisionTreeClassifier(
    max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE,
)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE,
)

dt_model, dt_metrics = score(dt, "DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)")
rf_model, rf_metrics = score(rf, "RandomForestClassifier(n_estimators=200, max_depth=20)")

# Persist both so the Admin Model Analytics page can compare them
joblib.dump(dt_model, os.path.join(OUT, "dt_model.joblib"))
joblib.dump(rf_model, os.path.join(OUT, "rf_model.joblib"))

# Winner (by F1 on the phishing class) becomes the deployment model
winner_name, winner_model = (
    ("RandomForest", rf_model)
    if rf_metrics["f1_score"] >= dt_metrics["f1_score"]
    else ("DecisionTree", dt_model)
)
print(f"\n>>> Winner (higher phishing-class F1): {winner_name}")

joblib.dump(winner_model, os.path.join(OUT, "deployment_model.joblib"))
joblib.dump(URL_ONLY_FEATURES, os.path.join(OUT, "deployment_features.joblib"))

DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)
               precision    recall  f1-score   support

Legitimate(0)     0.9853    0.9959    0.9906     27019
  Phishing(1)     0.9944    0.9800    0.9871     20104

     accuracy                         0.9891     47123
    macro avg     0.9898    0.9879    0.9889     47123
 weighted avg     0.9892    0.9891    0.9891     47123



RandomForestClassifier(n_estimators=200, max_depth=20)
               precision    recall  f1-score   support

Legitimate(0)     0.9940    0.9975    0.9958     27019
  Phishing(1)     0.9967    0.9919    0.9943     20104

     accuracy                         0.9951     47123
    macro avg     0.9953    0.9947    0.9950     47123
 weighted avg     0.9951    0.9951    0.9951     47123


>>> Winner (higher phishing-class F1): RandomForest


['../outputs\\deployment_features.joblib']

In [11]:
metrics_blob = {
    "winner": winner_name,
    "label_convention": {"0": "Legitimate", "1": "Phishing"},
    "positive_class": "Phishing",
    "decision_tree": dt_metrics,
    "random_forest": rf_metrics,
}
joblib.dump(metrics_blob, os.path.join(OUT, "deployment_metrics.joblib"))
with open(os.path.join(OUT, "evaluation_metrics.txt"), "w") as fh:
    fh.write(json.dumps(metrics_blob, indent=2))

bg = X_train.sample(1000, random_state=RANDOM_STATE).to_numpy()
np.save(os.path.join(OUT, "lime_background.npy"), bg)

In [12]:
print("\nSanity check on a few well-known URLs (0 = Legit, 1 = Phish):")
check_urls = [
    "https://chatgpt.com",
    "https://www.google.com/search?q=google+translate",
    "https://github.com/anthropics/anthropic-sdk-python",
    "https://en.wikipedia.org/wiki/Main_Page",
    "http://paypa1-secure-login.win/verify.php?id=aGVsbG8=&acct=443",
    "http://192.168.0.1/login.php?redirect=%2Fadmin",
]
for u in check_urls:
    fv = np.array([[compute_features(u, tld_lookup, char_lookup)[n]
                    for n in URL_ONLY_FEATURES]])
    p = winner_model.predict(fv)[0]
    prob = winner_model.predict_proba(fv)[0][1]
    print(f"  P(phish)={prob:.3f}  pred={int(p)}  {u}")

print("\nSaved to ./outputs:")
for f in [
    "deployment_model.joblib", "deployment_features.joblib",
    "deployment_metrics.joblib", "dt_model.joblib", "rf_model.joblib",
    "tld_probability_lookup.joblib", "char_probability_lookup.joblib",
    "lime_background.npy", "evaluation_metrics.txt",
]:
    print(f"  - {f}")



Sanity check on a few well-known URLs (0 = Legit, 1 = Phish):


  P(phish)=0.013  pred=0  https://chatgpt.com
  P(phish)=0.061  pred=0  https://www.google.com/search?q=google+translate
  P(phish)=0.598  pred=1  https://github.com/anthropics/anthropic-sdk-python


C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have 

  P(phish)=0.326  pred=0  https://en.wikipedia.org/wiki/Main_Page
  P(phish)=1.000  pred=1  http://paypa1-secure-login.win/verify.php?id=aGVsbG8=&acct=443
  P(phish)=1.000  pred=1  http://192.168.0.1/login.php?redirect=%2Fadmin

Saved to ./outputs:
  - deployment_model.joblib
  - deployment_features.joblib
  - deployment_metrics.joblib
  - dt_model.joblib
  - rf_model.joblib
  - tld_probability_lookup.joblib
  - char_probability_lookup.joblib
  - lime_background.npy
  - evaluation_metrics.txt


C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have 